In [1]:
from peft import LoraConfig, get_peft_model

c:\Users\weich\anaconda3\envs\LLM\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_path = "../model/huggingface_transformer_format"

tokenizer = AutoTokenizer.from_pretrained(model_path)

tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float32
)

model.config.pad_token_id = tokenizer.pad_token_id

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 146/146 [00:28<00:00,  5.18it/s, Materializing param=model.norm.weight]                              


In [4]:
from peft import get_peft_model

model = get_peft_model(model, lora_config)
model.print_trainable_parameters(
)

trainable params: 851,968 || all params: 1,236,666,368 || trainable%: 0.0689


In [5]:
# Load dataset
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="../data/train.json"
)

In [6]:
# Check Structure
print(dataset)
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['prompt', 'response'],
        num_rows: 2
    })
})
{'prompt': 'Explain machine learning in simple terms.', 'response': 'Machine learning is when computers learn patterns from data instead of being explicitly programmed.'}


In [7]:
def format_prompt(example):
    return f"""### Prompt:
{example['prompt']}

### Response:
{example['response']}"""

In [8]:
def tokenize(example):

    text = format_prompt(example)

    tokens = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=512
    )

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens


dataset = dataset.map(tokenize)

In [9]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="../lora_out",
    per_device_train_batch_size=1,
    num_train_epochs=3,
    logging_steps=1,
    save_strategy="epoch"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"]
)

trainer.train()

c:\Users\weich\anaconda3\envs\LLM\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,9.107330
2,8.853150
3,8.772132
4,8.542912
5,8.556808
6,8.355652


c:\Users\weich\anaconda3\envs\LLM\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\Users\weich\anaconda3\envs\LLM\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=6, training_loss=8.697997411092123, metrics={'train_runtime': 883.3172, 'train_samples_per_second': 0.007, 'train_steps_per_second': 0.007, 'total_flos': 17952732610560.0, 'train_loss': 8.697997411092123, 'epoch': 3.0})